In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, StructType, StructField, StringType, DoubleType
)

SOURCE_TABLE = "development_042_silver_sandbox.demand_forecast.game_attr_ai_bronz"
MAJOR_THEME_MIN_SCORE = 3
MAX_THEME_SCORE = 5.0

df = spark.table(SOURCE_TABLE)

required = {
    "game_name",
    "configurations",
    "canonical_theme_scores",
    "mechanic_scores",
}
missing = required - set(df.columns)
if missing:
    raise ValueError(
        f"Missing expected columns: {sorted(missing)}. "
        "Run printSchema() to check whether this table is at the scored/assigned stage."
    )

display(df.select("game_name", "configurations",
                  "canonical_theme_scores", "mechanic_scores").limit(5))

In [0]:
# Parse JSON strings if these fields are stored as strings.
# If Databricks already stores them as arrays of structs, leave them as-is.
config_schema = ArrayType(StructType([
    StructField("denom_cents", DoubleType(), True),
    StructField("ctc", StringType(), True),
    StructField("lines", DoubleType(), True),
    StructField("ways", DoubleType(), True),
    StructField("reels", DoubleType(), True),
]))

theme_schema = ArrayType(StructType([
    StructField("canonical_theme", StringType(), True),
    StructField("theme_score", DoubleType(), True),
]))

mechanic_schema = ArrayType(StructType([
    StructField("mechanic", StringType(), True),
    StructField("score_mean", DoubleType(), True),
    StructField("score_max", DoubleType(), True),
]))

def parse_array_if_string(frame, column, schema):
    if isinstance(frame.schema[column].dataType, StringType):
        return frame.withColumn(column, F.from_json(F.col(column), schema))
    return frame

df = parse_array_if_string(df, "configurations", config_schema)
df = parse_array_if_string(df, "canonical_theme_scores", theme_schema)
df = parse_array_if_string(df, "mechanic_scores", mechanic_schema)

# Add optional scalar base columns if this table only has configurations.
for c in ["denom_base", "ctc_base", "lines_base", "ways_base", "reels_base"]:
    if c not in df.columns:
        df = df.withColumn(c, F.lit(None).cast("double"))

# Safe numeric conversion for fields that may be strings.
for c in ["denom_base", "ctc_base", "lines_base", "ways_base", "reels_base"]:
    df = df.withColumn(f"_{c}_raw", F.expr(f"try_cast(`{c}` AS DOUBLE)"))

# Lowest denomination whose configuration has a positive CTC or ways value.
df = df.withColumn(
    "_eligible_configs",
    F.expr("""
        filter(
            configurations,
            x -> try_cast(x.denom_cents AS DOUBLE) > 0
                 AND (
                     try_cast(x.ctc AS DOUBLE) > 0
                     OR try_cast(x.ways AS DOUBLE) > 0
                 )
        )
    """)
).withColumn(
    "_lowest_eligible_denom",
    F.expr("""
        array_min(transform(
            _eligible_configs,
            x -> try_cast(x.denom_cents AS DOUBLE)
        ))
    """)
)

df = df.withColumn(
    "_chosen_denom",
    F.coalesce(
        F.when(F.col("_denom_base_raw") > 0, F.col("_denom_base_raw")),
        F.col("_lowest_eligible_denom")
    )
).withColumn(
    "_base_config",
    F.expr("""
        try_element_at(
            filter(
                configurations,
                x -> try_cast(x.denom_cents AS DOUBLE) = _chosen_denom
            ),
            1
        )
    """)
)

# Use existing valid base values first; otherwise use the selected configuration.
df = (
    df
    .withColumn(
        "denom_base_clean",
        F.coalesce(F.when(F.col("_denom_base_raw") > 0, F.col("_denom_base_raw")),
                   F.col("_chosen_denom"))
    )
    .withColumn(
        "ctc_base_clean",
        F.coalesce(F.when(F.col("_ctc_base_raw") > 0, F.col("_ctc_base_raw")),
                   F.expr("try_cast(_base_config.ctc AS DOUBLE)"))
    )
    .withColumn(
        "lines_base_clean",
        F.coalesce(F.when(F.col("_lines_base_raw") > 0, F.col("_lines_base_raw")),
                   F.col("_base_config.lines"))
    )
    .withColumn(
        "ways_base_clean",
        F.coalesce(F.when(F.col("_ways_base_raw") > 0, F.col("_ways_base_raw")),
                   F.col("_base_config.ways"))
    )
    .withColumn(
        "reels_base_clean",
        F.coalesce(F.when(F.col("_reels_base_raw") > 0, F.col("_reels_base_raw")),
                   F.col("_base_config.reels"))
    )
    .withColumn(
        "ways_or_lines_base",
        F.coalesce(F.col("ways_base_clean"), F.col("lines_base_clean"))
    )
)

# One audit flag: true if any output base value was filled from configurations.
df = df.withColumn(
    "base_values_imputed",
    (
        (F.col("_denom_base_raw").isNull() & F.col("_chosen_denom").isNotNull())
        | (F.col("_ctc_base_raw").isNull()
           & F.expr("try_cast(_base_config.ctc AS DOUBLE)").isNotNull())
        | (F.col("_ways_base_raw").isNull()
           & F.col("_base_config.ways").isNotNull())
        | (F.col("_lines_base_raw").isNull()
           & F.col("_ways_base_raw").isNull()
           & F.col("_base_config.lines").isNotNull())
        | (F.col("_reels_base_raw").isNull()
           & F.col("_base_config.reels").isNotNull())
    )
)

# Create safe, readable column names for each category.
def safe_name(value):
    return (
        value.lower()
        .replace("&", "and")
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace("'", "")
    )

theme_names = [
    r["label"] for r in (
        df.select(F.explode_outer("canonical_theme_scores").alias("s"))
          .select(F.col("s.canonical_theme").alias("label"))
          .where(F.col("label").isNotNull())
          .distinct()
          .collect()
    )
]

mechanic_names = [
    r["label"] for r in (
        df.select(F.explode_outer("mechanic_scores").alias("s"))
          .select(F.col("s.mechanic").alias("label"))
          .where(F.col("label").isNotNull())
          .distinct()
          .collect()
    )
]

# Each theme gets a column containing its theme_score.
theme_long = (
    df.select("game_name", F.explode_outer("canonical_theme_scores").alias("s"))
      .select("game_name",
              F.col("s.canonical_theme").alias("label"),
              F.col("s.theme_score").cast("double").alias("score"))
)
theme_wide = theme_long.groupBy("game_name").agg(*[
    F.max(F.when(F.col("label") == name, F.col("score")))
     .alias(f"theme__{safe_name(name)}")
    for name in theme_names
])

# Each mechanic gets score_mean and score_max columns.
mechanic_long = (
    df.select("game_name", F.explode_outer("mechanic_scores").alias("s"))
      .select("game_name",
              F.col("s.mechanic").alias("label"),
              F.col("s.score_mean").cast("double").alias("score_mean"),
              F.col("s.score_max").cast("double").alias("score_max"))
)
mechanic_wide = mechanic_long.groupBy("game_name").agg(*[
    expr
    for name in mechanic_names
    for expr in [
        F.max(F.when(F.col("label") == name, F.col("score_mean")))
         .alias(f"mechanic__{safe_name(name)}__score_mean"),
        F.max(F.when(F.col("label") == name, F.col("score_max")))
         .alias(f"mechanic__{safe_name(name)}__score_max"),
    ]
])

# Add None Trending when no theme reaches the configured major-theme threshold.
df = df.withColumn(
    "_max_theme_score",
    F.expr("""
        array_max(transform(
            filter(canonical_theme_scores, x -> x.theme_score IS NOT NULL),
            x -> cast(x.theme_score AS DOUBLE)
        ))
    """)
)

analysis_df = (
    df.select(
        "game_name",
        F.col("ctc_base_clean").alias("ctc_base"),
        F.col("ways_or_lines_base"),
        F.col("reels_base_clean").alias("reels_base"),
        F.col("denom_base_clean").alias("denom_base"),
        "base_values_imputed",
        "_max_theme_score",
    )
    .join(theme_wide, "game_name", "left")
    .join(mechanic_wide, "game_name", "left")
    .withColumn(
        "theme__none_trending",
        F.when(
            F.col("_max_theme_score").isNotNull()
            & (F.col("_max_theme_score") < MAJOR_THEME_MIN_SCORE),
            F.lit(MAX_THEME_SCORE)
        ).when(F.col("_max_theme_score").isNotNull(), F.lit(0.0))
    )
    .drop("_max_theme_score")
)

# Keep only the requested analysis fields.
score_columns = [
    c for c in analysis_df.columns
    if c.startswith("theme__") or c.startswith("mechanic__")
]
analysis_df = analysis_df.select(
    "game_name",
    "ctc_base",
    "ways_or_lines_base",
    "reels_base",
    "denom_base",
    "base_values_imputed",
    *score_columns,
)

display(analysis_df)